# Validação e configuração do ambiente

## 1. Objetivo

Configurar e validar o ambiente Databricks utilizado no projeto,
incluindo o Unity Catalog, os schemas das camadas Medallion e o
Volume responsável pelo armazenamento dos arquivos de entrada.

## 2. Arquitetura de armazenamento

O projeto utiliza o catálogo `workspace`, com schemas separados
por camada:

- `workspace.bronze`: dados ingeridos na camada Bronze.
- `workspace.silver`: dados tratados e validados na camada Silver,
  incluindo as tabelas de quarentena.
- `workspace.gold`: tabelas analíticas para consumo.
- `workspace.lakehouse`: área de armazenamento dos arquivos de entrada,
  por meio do Volume `raw`.

O Volume não representa uma camada Medallion; ele funciona como
área de aterrissagem (*landing zone*) dos arquivos CSV.


## 3. Verificação do catálogo

Consulta dos catálogos disponíveis e identificação do catálogo e
schema utilizados pela sessão atual.


In [0]:
%sql
SHOW CATALOGS;
USE CATALOG workspace;

In [0]:
print("Catálogo atual:", spark.sql("SELECT current_catalog()").first()[0])
print("Schema atual:", spark.sql("SELECT current_schema()").first()[0])

## 4. Criação dos schemas do projeto

Criação dos schemas correspondentes às camadas Bronze, Silver e Gold.
O schema `lakehouse` é mantido para armazenar o Volume `raw`.
A criação utiliza `IF NOT EXISTS` para permitir reexecuções sem erro
quando os schemas já estiverem disponíveis.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;
CREATE SCHEMA IF NOT EXISTS workspace.lakehouse;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

## 5. Configuração da área de entrada

Criação do Volume `raw` no schema `lakehouse`, utilizado como área
de entrada dos arquivos CSV originais.

Caminho do Volume:
`/Volumes/workspace/lakehouse/raw/`

Arquivos esperados:
- `clientes.csv`
- `contas.csv`
- `transacoes_pix.csv`


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.lakehouse.raw;

In [0]:
%sql
SHOW VOLUMES IN workspace.lakehouse;

## 6. Validação do ambiente Spark

Verificação da versão do Spark e execução de uma operação simples
para confirmar que a sessão está operacional.


In [0]:
print("Spark version:", spark.version)

df = spark.range(5)

display(df)

## 7. Validação de criação e visualização de DataFrame

Teste simples de criação de um DataFrame PySpark para verificar
a execução das operações básicas de processamento.


In [0]:
df_teste = spark.createDataFrame(
    [
        (1, "Cliente A"),
        (2, "Cliente B"),
        (3, "Cliente C")
    ],
    ["id_cliente", "nome"]
)

display(df_teste)

## 8. Validação dos diretórios de entrada

Verificação da disponibilidade do Volume utilizado para receber
os arquivos de origem. Os arquivos CSV devem estar presentes
antes da execução da ingestão Bronze.


In [0]:
raw_path = "/Volumes/workspace/lakehouse/raw/"

display(dbutils.fs.ls(raw_path))